In [ ]:
import pandas as pd

df = pd.read_csv("bias_clean.csv")

df = df.dropna(subset=["page_text"])
df["page_text"] = df["page_text"].astype(str)

df = df.drop_duplicates(
    subset=["page_text"]
).reset_index(drop=True)

print("Documents:", len(df))

Documents: 10832


In [ ]:
def chunk_text(text, chunk_size=150, overlap=30):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if len(chunk.strip()) > 50:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

rag_chunks = []

for doc_id, row in df.iterrows():

    chunks = chunk_text(row["page_text"])

    for chunk_id, chunk in enumerate(chunks):

        rag_chunks.append({
            "doc_id": doc_id,
            "chunk_id": chunk_id,
            "text": chunk
        })

rag_df = pd.DataFrame(rag_chunks)
print("Total chunks:", len(rag_df))

Total chunks: 79721


In [ ]:
import os
import numpy as np
import pandas as pd
import torch

# Install sentence-transformers if not already installed
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

# ============================================================
# GENERATE DOCUMENT EMBEDDINGS
# ============================================================


# ------------------------------------------------------------
# 1. Check GPU
# ------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print("DEVICE CHECK")
print("=" * 60)

print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 2. Select a smaller subset for development
# ------------------------------------------------------------

MAX_CHUNKS = len(rag_df)

if len(rag_df) > MAX_CHUNKS:

    rag_demo = rag_df.sample(
        n=MAX_CHUNKS,
        random_state=42
    ).reset_index(drop=True)

else:

    rag_demo = rag_df.copy().reset_index(drop=True)


print("\n" + "=" * 60)
print("RAG DATA")
print("=" * 60)

print("Total available chunks:", len(rag_df))
print("Chunks selected:", len(rag_demo))


# ------------------------------------------------------------
# 3. Paths for cached embeddings
# ------------------------------------------------------------

EMBEDDING_PATH = "/content/document_embeddings.npy"
CHUNKS_PATH = "/content/rag_chunks.csv"


# ------------------------------------------------------------
# 4. Load cached embeddings if they already exist
# ------------------------------------------------------------

if (
    os.path.exists(EMBEDDING_PATH)
    and os.path.exists(CHUNKS_PATH)
):

    print("\nCached embeddings found.")
    print("Loading existing embeddings...")

    embeddings = np.load(
        EMBEDDING_PATH
    )

    rag_demo = pd.read_csv(
        CHUNKS_PATH
    )

    print("Loaded embeddings:", embeddings.shape)
    print("Loaded chunks:", len(rag_demo))


# ------------------------------------------------------------
# 5. Otherwise generate embeddings
# ------------------------------------------------------------

else:

    print("\nNo cached embeddings found.")
    print("Loading Sentence Transformer...")

    embedding_model = SentenceTransformer(
        "all-MiniLM-L6-v2",
        device=device
    )

    texts = rag_demo["text"].astype(str).tolist()

    print("\nGenerating embeddings...")
    print("Number of texts:", len(texts))
    print("Batch size: 128")

    embeddings = embedding_model.encode(
        texts,
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # Make sure FAISS-compatible float32
    embeddings = embeddings.astype(
        "float32"
    )

    # --------------------------------------------------------
    # Save embeddings
    # --------------------------------------------------------

    np.save(
        EMBEDDING_PATH,
        embeddings
    )

    rag_demo.to_csv(
        CHUNKS_PATH,
        index=False
    )

    print("\nEmbeddings generated and saved.")


# ------------------------------------------------------------
# 6. Final information
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EMBEDDING COMPLETE")
print("=" * 60)

print("Embedding shape:", embeddings.shape)
print("Number of chunks:", len(rag_demo))
print("Embedding dimension:", embeddings.shape[1])

print("\nExample embedding:")
print(embeddings[0][:10])

print("\nFiles saved in Colab:")
print(EMBEDDING_PATH)
print(CHUNKS_PATH)

print("\nNext step:")
print("→ Build FAISS index")

DEVICE CHECK
CUDA available: False
Using device: cpu

RAG DATA
Total available chunks: 79720
Chunks selected: 79720

No cached embeddings found.
Loading Sentence Transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Generating embeddings...
Number of texts: 79720
Batch size: 128


Batches:   0%|          | 0/623 [00:00<?, ?it/s]

In [ ]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 95.6 MB/s eta 0:00:00


In [ ]:
# ============================================================
# BUILD FAISS INDEX + TEST SEMANTIC RETRIEVAL
# ============================================================

import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer


# ------------------------------------------------------------
# 1. Load saved embeddings and chunks
# ------------------------------------------------------------

EMBEDDING_PATH = "/content/document_embeddings.npy"
CHUNKS_PATH = "/content/rag_chunks.csv"

embeddings = np.load(
    EMBEDDING_PATH
).astype("float32")

rag_demo = pd.read_csv(
    CHUNKS_PATH
)

print("=" * 60)
print("LOADED DATA")
print("=" * 60)

print("Embeddings:", embeddings.shape)
print("Chunks:", len(rag_demo))


# ------------------------------------------------------------
# 2. Build FAISS index
# ------------------------------------------------------------

dimension = embeddings.shape[1]

# Inner Product + normalized vectors
# = cosine similarity

index = faiss.IndexFlatIP(
    dimension
)

index.add(
    embeddings
)

print("\n" + "=" * 60)
print("FAISS INDEX")
print("=" * 60)

print("Vector dimension:", dimension)
print("Vectors stored:", index.ntotal)


# ------------------------------------------------------------
# 3. Load Sentence Transformer
# ------------------------------------------------------------

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)

print("\nEmbedding model device:", device)


# ------------------------------------------------------------
# 4. Enter a user query
# ------------------------------------------------------------

query = "Are women underrepresented in AI?"

print("\n" + "=" * 60)
print("USER QUERY")
print("=" * 60)

print(query)


# ------------------------------------------------------------
# 5. Convert query → embedding model
# ------------------------------------------------------------

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True
)

query_embedding = query_embedding.astype(
    "float32"
)

print("\nQuery embedding shape:")
print(query_embedding.shape)


# ------------------------------------------------------------
# 6. Search FAISS
# ------------------------------------------------------------

TOP_K = 10

scores, indices = index.search(
    query_embedding,
    TOP_K
)


# ------------------------------------------------------------
# 7. Retrieve actual chunks
# ------------------------------------------------------------

retrieved_docs = rag_demo.iloc[
    indices[0]
].copy()

retrieved_docs["retrieval_score"] = scores[0]

retrieved_docs["rank"] = range(
    1,
    TOP_K + 1
)


# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TOP-K RETRIEVED DOCUMENTS")
print("=" * 60)

for _, row in retrieved_docs.iterrows():

    print("\n" + "-" * 60)

    print(
        f"Rank: {row['rank']}"
    )

    print(
        f"Similarity: "
        f"{row['retrieval_score']:.4f}"
    )

    print(
        f"Document ID: "
        f"{row['doc_id']}"
    )

    print(
        f"Chunk ID: "
        f"{row['chunk_id']}"
    )

    print("\nText:")

    print(
        str(row["text"])[:600]
    )

LOADED DATA
Embeddings: (79721, 384)
Chunks: 79721

FAISS INDEX
Vector dimension: 384
Vectors stored: 79721


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Embedding model device: cuda

USER QUERY
Are women underrepresented in AI?

Query embedding shape:
(1, 384)

TOP-K RETRIEVED DOCUMENTS

------------------------------------------------------------
Rank: 1
Similarity: 0.6717
Document ID: 6218
Chunk ID: 4

Text:
disadvantaged in the labour market already”. Evidence of gender and racial bias in AI-powered hiring processes has emerged in recent years and the issue of AI-driven prejudice has become one of the key safety concerns about the technology.

------------------------------------------------------------
Rank: 2
Similarity: 0.6508
Document ID: 6218
Chunk ID: 3

Text:
one, with the share of firms that had adopted AI remaining in the single digits, due in part to issues related to cost and workforce skills. The OECD added that while AI had the potential to eliminate boring or dangerous tasks and create interesting ones instead, firms were open about the fact that a main motivation for investing in AI was improving worker performance a

In [ ]:
import os

print(os.listdir())

['.config', 'document_embeddings.npy', 'bias_clean.csv', 'rag_chunks.csv', 'sample_data']


In [ ]:
import pandas as pd

df = pd.read_csv("bias_clean.csv")

print("Dataset shape:", df.shape)

Dataset shape: (10832, 7)


In [ ]:
print(df.columns)

Index(['url', 'topic', 'date', 'title', 'site', 'bias', 'page_text'], dtype='object')


In [ ]:
df.head()


,url,topic,date,title,site,bias,page_text
0,https://www.npr.org/2021/01/01/952336030/what-...,general-news,2021-01-01,"What Got Us Through 2020? For Many, It Was Hob...",NPR (Online News),leaning-left,"What Got Us Through 2020? For Many, It Was Hob..."
1,https://www.cbsnews.com/news/new-years-eve-gat...,general-news,2020-12-31,New Year's Eve gatherings could accelerate COV...,CBS News (Online),leaning-left,New Year's Eve gatherings could accelerate COV...
2,https://nypost.com/2020/12/31/heres-how-countr...,general-news,2020-12-31,Here’s how countries around the world are ring...,New York Post (News),leaning-right,Here’s how countries around the world are ring...
3,https://www.theguardian.com/world/2020/dec/31/...,coronavirus,2020-12-31,Wisconsin health worker 'deliberately spoiled ...,The Guardian,left,Police in Wisconsin said on Thursday evening t...
4,https://www.foxnews.com/us/wis-hospital-intent...,coronavirus,2020-12-31,Wisconsin hospital employee 'intentionally' re...,Fox News Digital,right,A Wisconsin-based hospital on Wednesday said a...


In [ ]:
print(df.isnull().sum())

url          0
topic        0
date         0
title        0
site         0
bias         0
page_text    0
dtype: int64


In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10832 entries, 0 to 10831
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   url        10832 non-null  object
 1   topic      10832 non-null  object
 2   date       10832 non-null  object
 3   title      10832 non-null  object
 4   site       10832 non-null  object
 5   bias       10832 non-null  object
 6   page_text  10832 non-null  object
dtypes: object(7)
memory usage: 592.5+ KB
None


In [ ]:
texts = df["page_text"].astype(str).tolist()

print("Number of documents:", len(texts))
print("First document:")
print(texts[0])

Number of documents: 10832
First document:
What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals
What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals
It was an eventful year to say the least. It began with a historic impeachment, and then the global pandemic, a reckoning around racial injustice, a tumultuous election and even more unlikely news.
The events of 2020, however significant, had an effect on everyone. NPR's Morning Edition wanted to know what helped get people through the tough times — and hundreds responded on Twitter.
Some said they turned to writing letters, creating art or starting a new hobby. Others relied on friends and family and mindless escapist TV shows. And many said it was four-legged friends and good wine that got them through.
Wine and my dogs.
— JasonX(@JasonX17428071)December 21,2020
Sahil Swali from Albany, N.Y., wanted to help people feel connected, especially those who were more isolated. It ins

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
embeddings = model.encode(
    texts,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/339 [00:00<?, ?it/s]

Embedding shape: (10832, 384)


In [ ]:
print(df["bias"].value_counts())


bias
right            2986
leaning-left     2951
left             1930
leaning-right    1487
center           1478
Name: count, dtype: int64


In [ ]:
rag_demo

,doc_id,chunk_id,text
0,0,0,"What Got Us Through 2020? For Many, It Was Hob..."
1,0,1,it was four-legged friends and good wine that ...
2,0,2,off in smiles and making people feel less lone...
3,0,3,do this activity that I hadn't done in years a...
4,0,4,the Crouton Crew grew and now the account dedi...
...,...,...,...
79716,10830,3,"vengeance for Donald Trump,"" Biden-Harris 2024..."
79717,10831,0,CNN issued several controversial presidential ...
79718,10831,1,rules: The 90-minute debate will include two c...
79719,10831,2,tools at their disposal to enforce timing and ...


In [ ]:
print(rag_demo.columns.tolist())
print(rag_demo.head())

['doc_id', 'chunk_id', 'text']
   doc_id  chunk_id                                               text
0       0         0  What Got Us Through 2020? For Many, It Was Hob...
1       0         1  it was four-legged friends and good wine that ...
2       0         2  off in smiles and making people feel less lone...
3       0         3  do this activity that I hadn't done in years a...
4       0         4  the Crouton Crew grew and now the account dedi...


In [ ]:
print(df.columns.tolist())

['url', 'topic', 'date', 'title', 'site', 'bias', 'page_text']


In [ ]:
print(df["bias"].value_counts())

bias
right            2986
leaning-left     2951
left             1930
leaning-right    1487
center           1478
Name: count, dtype: int64


In [ ]:
print(rag_demo.columns.tolist())

['doc_id', 'chunk_id', 'text']


In [ ]:
print(rag_demo.head())

   doc_id  chunk_id                                               text
0       0         0  What Got Us Through 2020? For Many, It Was Hob...
1       0         1  it was four-legged friends and good wine that ...
2       0         2  off in smiles and making people feel less lone...
3       0         3  do this activity that I hadn't done in years a...
4       0         4  the Crouton Crew grew and now the account dedi...


In [ ]:
# ============================================================
# CONNECT RETRIEVED DOCUMENTS TO BIAS LABELS
# ============================================================

# Create a mapping from document ID to its bias label
bias_map = df.set_index("url")["bias"].to_dict()

# Check what doc_id looks like
print("Sample doc_ids:")
print(rag_demo["doc_id"].head())

print("\nSample original URLs:")
print(df["url"].head())

Sample doc_ids:
0    0
1    0
2    0
3    0
4    0
Name: doc_id, dtype: int64

Sample original URLs:
0    https://www.npr.org/2021/01/01/952336030/what-...
1    https://www.cbsnews.com/news/new-years-eve-gat...
2    https://nypost.com/2020/12/31/heres-how-countr...
3    https://www.theguardian.com/world/2020/dec/31/...
4    https://www.foxnews.com/us/wis-hospital-intent...
Name: url, dtype: object


In [ ]:
print("Number of RAG documents:", rag_demo["doc_id"].nunique())
print("Number of original documents:", df["url"].nunique())

print("\nMatching doc IDs:")

matches = rag_demo["doc_id"].isin(df["url"])

print("Matched:", matches.sum())
print("Total chunks:", len(rag_demo))
print("Match percentage:", matches.mean() * 100)

Number of RAG documents: 10832
Number of original documents: 10832

Matching doc IDs:
Matched: 0
Total chunks: 79721
Match percentage: 0.0


In [ ]:
print("RAG doc_id examples:")
print(rag_demo["doc_id"].head(20).tolist())

print("\nOriginal dataframe index examples:")
print(df.index[:20].tolist())

print("\nOriginal URL examples:")
print(df["url"].head(5).tolist())

print("\nOriginal title examples:")
print(df["title"].head(5).tolist())

RAG doc_id examples:
[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3]

Original dataframe index examples:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

Original URL examples:
['https://www.npr.org/2021/01/01/952336030/what-got-us-through-2020-for-many-it-was-hobbies-relationships-and-4-legged-pals', 'https://www.cbsnews.com/news/new-years-eve-gatherings-covid-19-spread/', 'https://nypost.com/2020/12/31/heres-how-countries-around-the-world-are-ringing-in-2021/', 'https://www.theguardian.com/world/2020/dec/31/wisconsin-health-worker-deliberately-spoiled-500-covid-vaccine-doses', 'https://www.foxnews.com/us/wis-hospital-intentionally-covid-19-vaccine']

Original title examples:
['What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals', 'New Year\'s Eve gatherings could accelerate COVID-19 spread: "We cannot let our guard down"', 'Here’s how countries around the world are ringing in 2021', "Wisconsin health worker 'delibera

In [ ]:
print(rag_demo.head(10).to_string())

   doc_id  chunk_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        text
0       

In [ ]:
print(rag_demo["doc_id"].head(20).tolist())
print(df.index[:20].tolist())
print(df["url"].head(5).tolist())

[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
['https://www.npr.org/2021/01/01/952336030/what-got-us-through-2020-for-many-it-was-hobbies-relationships-and-4-legged-pals', 'https://www.cbsnews.com/news/new-years-eve-gatherings-covid-19-spread/', 'https://nypost.com/2020/12/31/heres-how-countries-around-the-world-are-ringing-in-2021/', 'https://www.theguardian.com/world/2020/dec/31/wisconsin-health-worker-deliberately-spoiled-500-covid-vaccine-doses', 'https://www.foxnews.com/us/wis-hospital-intentionally-covid-19-vaccine']


In [ ]:
print(rag_demo.head(10).to_string())

   doc_id  chunk_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        text
0       

In [ ]:
print("RAG doc_id examples:")
print(rag_demo["doc_id"].head(20).tolist())

print("\nOriginal URL examples:")
print(df["url"].head(5).tolist())

print("\nOriginal index examples:")
print(df.index[:20].tolist())

RAG doc_id examples:
[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3]

Original URL examples:
['https://www.npr.org/2021/01/01/952336030/what-got-us-through-2020-for-many-it-was-hobbies-relationships-and-4-legged-pals', 'https://www.cbsnews.com/news/new-years-eve-gatherings-covid-19-spread/', 'https://nypost.com/2020/12/31/heres-how-countries-around-the-world-are-ringing-in-2021/', 'https://www.theguardian.com/world/2020/dec/31/wisconsin-health-worker-deliberately-spoiled-500-covid-vaccine-doses', 'https://www.foxnews.com/us/wis-hospital-intentionally-covid-19-vaccine']

Original index examples:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [ ]:
print(rag_demo.head(10).to_string())

   doc_id  chunk_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        text
0       

In [ ]:
print(df[["url", "title", "bias"]].head(10).to_string())

                                                                                                                                                           url                                                                                       title           bias
0                                    https://www.npr.org/2021/01/01/952336030/what-got-us-through-2020-for-many-it-was-hobbies-relationships-and-4-legged-pals         What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals   leaning-left
1                                                                                       https://www.cbsnews.com/news/new-years-eve-gatherings-covid-19-spread/  New Year's Eve gatherings could accelerate COVID-19 spread: "We cannot let our guard down"   leaning-left
2                                                                      https://nypost.com/2020/12/31/heres-how-countries-around-the-world-are-ringing-in-2021/                                   Here’s ho

In [ ]:
print(rag_demo["doc_id"].head(20).tolist())

[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 3, 3, 3]


In [ ]:
print(rag_demo.head(10).to_string())

   doc_id  chunk_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        text
0       

In [ ]:
print(df[["url", "title", "bias"]].head(10).to_string())

                                                                                                                                                           url                                                                                       title           bias
0                                    https://www.npr.org/2021/01/01/952336030/what-got-us-through-2020-for-many-it-was-hobbies-relationships-and-4-legged-pals         What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals   leaning-left
1                                                                                       https://www.cbsnews.com/news/new-years-eve-gatherings-covid-19-spread/  New Year's Eve gatherings could accelerate COVID-19 spread: "We cannot let our guard down"   leaning-left
2                                                                      https://nypost.com/2020/12/31/heres-how-countries-around-the-world-are-ringing-in-2021/                                   Here’s ho

In [ ]:
# ============================================================
# VERIFY DOC_ID → ORIGINAL DATASET MAPPING
# ============================================================

sample_ids = rag_demo["doc_id"].head(10).tolist()

for doc_id in sample_ids:
    print(
        "doc_id:", doc_id,
        "| title:", df.iloc[doc_id]["title"],
        "| bias:", df.iloc[doc_id]["bias"]
    )

doc_id: 0 | title: What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals | bias: leaning-left
doc_id: 0 | title: What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals | bias: leaning-left
doc_id: 0 | title: What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals | bias: leaning-left
doc_id: 0 | title: What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals | bias: leaning-left
doc_id: 0 | title: What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals | bias: leaning-left
doc_id: 0 | title: What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals | bias: leaning-left
doc_id: 0 | title: What Got Us Through 2020? For Many, It Was Hobbies, Relationships And 4-Legged Pals | bias: leaning-left
doc_id: 1 | title: New Year's Eve gatherings could accelerate COVID-19 spread: "We cannot let our guard down" | bias: leaning-left
d

In [ ]:
# ============================================================
# ADD ORIGINAL BIAS LABEL TO RAG CHUNKS
# ============================================================

rag_demo["bias"] = rag_demo["doc_id"].apply(
    lambda x: df.iloc[int(x)]["bias"]
)

print(rag_demo.head(10))

   doc_id  chunk_id                                               text  \
0       0         0  What Got Us Through 2020? For Many, It Was Hob...   
1       0         1  it was four-legged friends and good wine that ...   
2       0         2  off in smiles and making people feel less lone...   
3       0         3  do this activity that I hadn't done in years a...   
4       0         4  the Crouton Crew grew and now the account dedi...   
5       0         5  of his for years — beekeeping. I became a beek...   
6       0         6  it's up to us to wake up and pay attention and...   
7       1         0  New Year's Eve gatherings could accelerate COV...   
8       1         1  December 23 to January 3 to celebrate the holi...   
9       1         2  down on large gatherings. In Newark, New Jerse...   

           bias  
0  leaning-left  
1  leaning-left  
2  leaning-left  
3  leaning-left  
4  leaning-left  
5  leaning-left  
6  leaning-left  
7  leaning-left  
8  leaning-left  
9  le

In [ ]:
print(rag_demo["bias"].value_counts())

bias
leaning-left     24741
right            19636
left             16475
center           11094
leaning-right     7775
Name: count, dtype: int64


In [ ]:
query = "Are women underrepresented in AI?"

In [ ]:
scores, indices = index.search(
    query_embedding,
    TOP_K
)

In [ ]:
# ============================================================
# RETRIEVE TOP-K DOCUMENTS
# ============================================================

retrieved_docs = rag_demo.iloc[indices[0]].copy()

retrieved_docs["retrieval_score"] = scores[0]

retrieved_docs["rank"] = range(1, TOP_K + 1)

print(
    retrieved_docs[
        ["rank", "doc_id", "chunk_id", "retrieval_score", "bias"]
    ]
)

       rank  doc_id  chunk_id  retrieval_score           bias
44375     1    6218         4         0.671714           left
44374     2    6218         3         0.650771           left
49076     3    6861         0         0.545302         center
44377     4    6219         1         0.544573  leaning-right
63717     5    8795         6         0.532482   leaning-left
14844     6    1955        14         0.532296   leaning-left
44379     7    6219         3         0.523930  leaning-right
44371     8    6218         0         0.522245           left
44376     9    6219         0         0.516462  leaning-right
63728    10    8796         5         0.514190         center


In [ ]:
# ============================================================
# BIAS DETECTOR
# ============================================================

bias_counts = retrieved_docs["bias"].value_counts()

print("=" * 60)
print("BIAS DETECTOR")
print("=" * 60)

print("\nRetrieved documents:", len(retrieved_docs))

print("\nBias distribution:")
print(bias_counts)

BIAS DETECTOR

Retrieved documents: 10

Bias distribution:
bias
left             3
leaning-right    3
center           2
leaning-left     2
Name: count, dtype: int64


In [ ]:
bias_percentages = (
    retrieved_docs["bias"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nBias percentages:")
print(bias_percentages)



Bias percentages:
bias
left             30.0
leaning-right    30.0
center           20.0
leaning-left     20.0
Name: proportion, dtype: float64


In [ ]:
dominant_bias = bias_percentages.idxmax()
dominant_percentage = bias_percentages.max()

print("\nDominant source bias:", dominant_bias)
print("Percentage:", dominant_percentage, "%")


Dominant source bias: left
Percentage: 30.0 %


In [ ]:
# ============================================================
# WEIGHTED BIAS DETECTION
# ============================================================

scores = retrieved_docs["retrieval_score"].values

# Convert similarity scores into positive weights
exp_scores = np.exp(scores - np.max(scores))

weights = exp_scores / exp_scores.sum()

retrieved_docs["weight"] = weights

weighted_bias = (
    retrieved_docs
    .groupby("bias")["weight"]
    .sum()
    .sort_values(ascending=False)
)

weighted_bias_percentage = (
    weighted_bias * 100
).round(2)

print("=" * 60)
print("WEIGHTED BIAS DISTRIBUTION")
print("=" * 60)

print(weighted_bias_percentage)

WEIGHTED BIAS DISTRIBUTION
bias
left             31.860001
leaning-right    29.160000
leaning-left     19.520000
center           19.469999
Name: weight, dtype: float32


In [ ]:
final_source_bias = weighted_bias_percentage.idxmax()

print("\nFinal detected source bias:", final_source_bias)
print(
    "Weighted percentage:",
    weighted_bias_percentage[final_source_bias],
    "%"
)


Final detected source bias: left
Weighted percentage: 31.86 %


In [ ]:
# ============================================================
# BIAS MITIGATION
# ============================================================

# Convert weighted bias percentages into values between 0 and 1
bias_weights = weighted_bias_percentage / 100

print("Bias weights:")
print(bias_weights)

Bias weights:
bias
left             0.3186
leaning-right    0.2916
leaning-left     0.1952
center           0.1947
Name: weight, dtype: float32


In [ ]:
retrieved_docs["bias_weight"] = (
    retrieved_docs["bias"]
    .map(bias_weights)
    .fillna(0)
)

print(
    retrieved_docs[
        ["bias", "retrieval_score", "bias_weight"]
    ]
)

                bias  retrieval_score  bias_weight
44375           left         0.671714       0.3186
44374           left         0.650771       0.3186
49076         center         0.545302       0.1947
44377  leaning-right         0.544573       0.2916
63717   leaning-left         0.532482       0.1952
14844   leaning-left         0.532296       0.1952
44379  leaning-right         0.523930       0.2916
44371           left         0.522245       0.3186
44376  leaning-right         0.516462       0.2916
63728         center         0.514190       0.1947


In [ ]:
ALPHA = 0.5

retrieved_docs["mitigation_score"] = (
    retrieved_docs["retrieval_score"]
    * (
        1 - ALPHA * retrieved_docs["bias_weight"]
    )
)

In [ ]:
retrieved_docs = retrieved_docs.sort_values(
    by="mitigation_score",
    ascending=False
).reset_index(drop=True)

In [ ]:
print(
    retrieved_docs[
        [
            "bias",
            "retrieval_score",
            "bias_weight",
            "mitigation_score"
        ]
    ]
)

            bias  retrieval_score  bias_weight  mitigation_score
0           left         0.671714       0.3186          0.564710
1           left         0.650771       0.3186          0.547103
2         center         0.545302       0.1947          0.492217
3   leaning-left         0.532482       0.1952          0.480511
4   leaning-left         0.532296       0.1952          0.480344
5  leaning-right         0.544573       0.2916          0.465174
6         center         0.514190       0.1947          0.464134
7  leaning-right         0.523930       0.2916          0.447541
8  leaning-right         0.516462       0.2916          0.441162
9           left         0.522245       0.3186          0.439051


In [ ]:
retrieved_docs.head(5)

,doc_id,chunk_id,text,bias,retrieval_score,rank,weight,bias_weight,mitigation_score
0,6218,4,disadvantaged in the labour market already”. E...,left,0.671714,1,0.112167,0.3186,0.564710
1,6218,3,"one, with the share of firms that had adopted ...",left,0.650771,2,0.109842,0.3186,0.547103
2,6861,0,"On this International Women’s Day in 2023, the...",center,0.545302,3,0.098848,0.1947,0.492217
3,8795,6,with AI advancements in the workforce. “We hav...,leaning-left,0.532482,5,0.097588,0.1952,0.480511
4,1955,14,jobs last year. All Americans can be proud tha...,leaning-left,0.532296,6,0.097570,0.1952,0.480344


In [ ]:
# ============================================================
# SELECT BALANCED EVIDENCE
# ============================================================

selected_docs = (
    retrieved_docs
    .drop_duplicates(
        subset=["bias"]
    )
    .head(5)
    .copy()
)

print("Selected evidence:")
print(
    selected_docs[
        [
            "bias",
            "retrieval_score",
            "mitigation_score"
        ]
    ]
)

Selected evidence:
            bias  retrieval_score  mitigation_score
0           left         0.671714          0.564710
2         center         0.545302          0.492217
3   leaning-left         0.532482          0.480511
5  leaning-right         0.544573          0.465174


In [ ]:
for i, row in selected_docs.iterrows():

    print("\n" + "=" * 70)
    print("EVIDENCE", i + 1)
    print("=" * 70)

    print("Bias category:", row["bias"])
    print("Retrieval score:", row["retrieval_score"])
    print("Mitigation score:", row["mitigation_score"])

    print("\nText:")
    print(row["text"])


EVIDENCE 1
Bias category: left
Retrieval score: 0.671714186668396
Mitigation score: 0.5647101402282715

Text:
disadvantaged in the labour market already”. Evidence of gender and racial bias in AI-powered hiring processes has emerged in recent years and the issue of AI-driven prejudice has become one of the key safety concerns about the technology.

EVIDENCE 3
Bias category: center
Retrieval score: 0.545302152633667
Mitigation score: 0.49221697449684143

Text:
On this International Women’s Day in 2023, the “glass ceiling” is alive and well, blocking women from advancement across the corporate world. This is true at Fortune 500 companies, where they experience slower career climbs than their male peers, among executives in general—where more are excluded from informal networks and mentors—and among MBA graduates’ careers, where women end up paid less. That’s all despite having similar qualifications, skills and experience. Smart, savvy businesses should hire more women for those top job

In [ ]:
# ============================================================
# BUILD LLM CONTEXT
# ============================================================

context_parts = []

for i, row in selected_docs.iterrows():

    context_parts.append(
        f"""
Evidence {i + 1}
Source bias category: {row["bias"]}

{row["text"]}
"""
    )

context = "\n".join(context_parts)

print(context)


Evidence 1
Source bias category: left

disadvantaged in the labour market already”. Evidence of gender and racial bias in AI-powered hiring processes has emerged in recent years and the issue of AI-driven prejudice has become one of the key safety concerns about the technology.


Evidence 3
Source bias category: center

On this International Women’s Day in 2023, the “glass ceiling” is alive and well, blocking women from advancement across the corporate world. This is true at Fortune 500 companies, where they experience slower career climbs than their male peers, among executives in general—where more are excluded from informal networks and mentors—and among MBA graduates’ careers, where women end up paid less. That’s all despite having similar qualifications, skills and experience. Smart, savvy businesses should hire more women for those top jobs, for reasons that go beyond even fairness. That’s because a wealth of research suggests that hiring women for the top perch pays off—a hidde

In [ ]:
# ============================================================
# CREATE LLM PROMPT
# ============================================================

prompt = f"""
You are a bias-aware research assistant.

User question:
{query}

The following evidence was retrieved from multiple
sources and re-ranked to reduce source dominance.

Evidence:
{context}

Instructions:

1. Answer the user's question using the evidence.
2. Give a balanced and factual response.
3. Do not treat any single source category as automatically correct.
4. Do not introduce unsupported claims.
5. If the evidence is conflicting, explain the disagreement.
6. If the evidence is insufficient, clearly say so.
7. Avoid reinforcing stereotypes or biased assumptions.

Provide a clear final answer.
"""

print(prompt)


You are a bias-aware research assistant.

User question:
Are women underrepresented in AI?

The following evidence was retrieved from multiple
sources and re-ranked to reduce source dominance.

Evidence:

Evidence 1
Source bias category: left

disadvantaged in the labour market already”. Evidence of gender and racial bias in AI-powered hiring processes has emerged in recent years and the issue of AI-driven prejudice has become one of the key safety concerns about the technology.


Evidence 3
Source bias category: center

On this International Women’s Day in 2023, the “glass ceiling” is alive and well, blocking women from advancement across the corporate world. This is true at Fortune 500 companies, where they experience slower career climbs than their male peers, among executives in general—where more are excluded from informal networks and mentors—and among MBA graduates’ careers, where women end up paid less. That’s all despite having similar qualifications, skills and experience. S

In [ ]:
# ============================================================
# CHECK BIAS MITIGATION RESULTS
# ============================================================

print("=" * 70)
print("BIAS MITIGATION RESULTS")
print("=" * 70)

print("\nTotal retrieved documents:", len(retrieved_docs))
print("Selected documents:", len(selected_docs))

print("\nSelected evidence ranking:")

print(
    selected_docs[
        [
            "doc_id",
            "chunk_id",
            "bias",
            "retrieval_score",
            "bias_weight",
            "mitigation_score"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("SELECTED EVIDENCE")
print("=" * 70)

for i, row in selected_docs.iterrows():

    print(f"\n--- Evidence {i + 1} ---")
    print("Bias category:", row["bias"])
    print("Retrieval score:", round(row["retrieval_score"], 4))
    print("Bias weight:", round(row["bias_weight"], 4))
    print("Mitigation score:", round(row["mitigation_score"], 4))

    print("\nText:")
    print(row["text"])

BIAS MITIGATION RESULTS

Total retrieved documents: 10
Selected documents: 4

Selected evidence ranking:
 doc_id  chunk_id          bias  retrieval_score  bias_weight  mitigation_score
   6218         4          left         0.671714       0.3186          0.564710
   6861         0        center         0.545302       0.1947          0.492217
   8795         6  leaning-left         0.532482       0.1952          0.480511
   6219         1 leaning-right         0.544573       0.2916          0.465174

SELECTED EVIDENCE

--- Evidence 1 ---
Bias category: left
Retrieval score: 0.6717
Bias weight: 0.3186
Mitigation score: 0.5647

Text:
disadvantaged in the labour market already”. Evidence of gender and racial bias in AI-powered hiring processes has emerged in recent years and the issue of AI-driven prejudice has become one of the key safety concerns about the technology.

--- Evidence 3 ---
Bias category: center
Retrieval score: 0.5453
Bias weight: 0.1947
Mitigation score: 0.4922

Text:
On

### Bias Mitigation Result Report

In [ ]:
# ============================================================
# QUERY → EMBEDDING → FAISS RETRIEVAL
# ============================================================

query = "Are women underrepresented in AI?"

print("=" * 60)
print("USER QUERY")
print("=" * 60)
print(query)


# ------------------------------------------------------------
# 1. Convert query into an embedding
# ------------------------------------------------------------

query_embedding = embedding_model.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True
)

query_embedding = query_embedding.astype("float32")

print("\nQuery embedding shape:", query_embedding.shape)


# ------------------------------------------------------------
# 2. Search FAISS
# ------------------------------------------------------------

TOP_K = 10

scores, indices = index.search(
    query_embedding,
    TOP_K
)

print("\nFAISS search completed.")
print("Retrieved documents:", TOP_K)


# ------------------------------------------------------------
# 3. Get the actual chunks
# ------------------------------------------------------------

retrieved_docs = rag_demo.iloc[
    indices[0]
].copy()

retrieved_docs["retrieval_score"] = scores[0]

retrieved_docs = retrieved_docs.reset_index(drop=True)


# ------------------------------------------------------------
# 4. Display retrieval summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("RETRIEVAL RESULTS")
print("=" * 60)

print(
    retrieved_docs[
        [
            "doc_id",
            "chunk_id",
            "retrieval_score"
        ]
    ].to_string(index=False)
)

USER QUERY
Are women underrepresented in AI?

Query embedding shape: (1, 384)

FAISS search completed.
Retrieved documents: 10

RETRIEVAL RESULTS
 doc_id  chunk_id  retrieval_score
   6218         4         0.671714
   6218         3         0.650771
   6861         0         0.545302
   6219         1         0.544573
   8795         6         0.532482
   1955        14         0.532296
   6219         3         0.523930
   6218         0         0.522245
   6219         0         0.516462
   8796         5         0.514190


In [ ]:
# ============================================================
# INSPECT RETRIEVED EVIDENCE
# ============================================================

print("=" * 80)
print("RETRIEVED EVIDENCE")
print("=" * 80)

for i, row in retrieved_docs.iterrows():

    print("\n" + "-" * 80)
    print(f"RANK: {i + 1}")
    print("-" * 80)

    print("Document ID:", row["doc_id"])
    print("Chunk ID:", row["chunk_id"])
    print("Retrieval score:", round(row["retrieval_score"], 4))

    if "bias" in retrieved_docs.columns:
        print("Bias category:", row["bias"])

    print("\nTEXT:")
    print(row["text"])

RETRIEVED EVIDENCE

--------------------------------------------------------------------------------
RANK: 1
--------------------------------------------------------------------------------
Document ID: 6218
Chunk ID: 4
Retrieval score: 0.6717
Bias category: left

TEXT:
disadvantaged in the labour market already”. Evidence of gender and racial bias in AI-powered hiring processes has emerged in recent years and the issue of AI-driven prejudice has become one of the key safety concerns about the technology.

--------------------------------------------------------------------------------
RANK: 2
--------------------------------------------------------------------------------
Document ID: 6218
Chunk ID: 3
Retrieval score: 0.6508
Bias category: left

TEXT:
one, with the share of firms that had adopted AI remaining in the single digits, due in part to issues related to cost and workforce skills. The OECD added that while AI had the potential to eliminate boring or dangerous tasks and create

In [ ]:
# Bias scores and mitigation scores

standard_top_5 = retrieved_docs.sort_values(
    'retrieval_score', ascending=False
).head(5)

selected_docs = selected_docs.sort_values(
    'mitigation_score', ascending=False
)

print(f"Query: {query}\n")

for i, row in selected_docs.iterrows():
    print(f"Source: {row['bias'].upper()}")
    print(f"Relevance Score: {row['retrieval_score']:.4f}")
    print(f"Mitigation Score: {row['mitigation_score']:.4f}")
    print(f"Result: {row['text']}\n")

Query: Are women underrepresented in AI?

Source: LEFT
Relevance Score: 0.6717
Mitigation Score: 0.5647
Result: disadvantaged in the labour market already”. Evidence of gender and racial bias in AI-powered hiring processes has emerged in recent years and the issue of AI-driven prejudice has become one of the key safety concerns about the technology.

Source: CENTER
Relevance Score: 0.5453
Mitigation Score: 0.4922
Result: On this International Women’s Day in 2023, the “glass ceiling” is alive and well, blocking women from advancement across the corporate world. This is true at Fortune 500 companies, where they experience slower career climbs than their male peers, among executives in general—where more are excluded from informal networks and mentors—and among MBA graduates’ careers, where women end up paid less. That’s all despite having similar qualifications, skills and experience. Smart, savvy businesses should hire more women for those top jobs, for reasons that go beyond even fairn

In [ ]:
#Final Response and results

standard_top_5 = retrieved_docs.sort_values(
    'retrieval_score', ascending=False
).head(5)

std_biases = standard_top_5['bias'].unique().tolist()
mit_biases = selected_docs['bias'].unique().tolist()

report_output = f"""
Query: {query}

Result:
"""

for i, row in selected_docs.iterrows():
    report_output += f"""
{row['text']}

"""

print(report_output)


Query: Are women underrepresented in AI?

Result:

disadvantaged in the labour market already”. Evidence of gender and racial bias in AI-powered hiring processes has emerged in recent years and the issue of AI-driven prejudice has become one of the key safety concerns about the technology.


On this International Women’s Day in 2023, the “glass ceiling” is alive and well, blocking women from advancement across the corporate world. This is true at Fortune 500 companies, where they experience slower career climbs than their male peers, among executives in general—where more are excluded from informal networks and mentors—and among MBA graduates’ careers, where women end up paid less. That’s all despite having similar qualifications, skills and experience. Smart, savvy businesses should hire more women for those top jobs, for reasons that go beyond even fairness. That’s because a wealth of research suggests that hiring women for the top perch pays off—a hidden, discriminated-against, key